In [1]:
# %pip install -r ../requirements_databricks.txt
# dbutils.library.restartPython()

In [2]:
# read csv data
# treat all of them the same -> function?
# we need to clean:
    # duplicates?
    # shape, type of data? -> floats, datetime, strings
    # columns plr_key: transform to strings and add "0" if necessary
    # round numbers eventually
    # missing data: delete content but keep plr
# put everything together in one big table???
# keep only columns we really want and need and do feature engineering! -> think about this first, back it up with literature

In [3]:
# import libraries for data cleaning and file handling

import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
# define path to data folder inside github repo

DATA_DIR = Path("../data")

files = {
    "population_2020": DATA_DIR / "population_2021.csv",
    "population_2021": DATA_DIR / "population_2022.csv",
    "population_2022": DATA_DIR / "population_2023.csv",
    "population_2023": DATA_DIR / "population_2024.csv",
    "population_2024": DATA_DIR / "population_2025.csv",
    "population_2025": DATA_DIR / "population_2026.csv",
    "fluctuation_2020": DATA_DIR / "fluctuation_2021.csv",
    "fluctuation_2021": DATA_DIR / "fluctuation_2022.csv",
    "fluctuation_2022": DATA_DIR / "fluctuation_2023.csv",
    "fluctuation_2023": DATA_DIR / "fluctuation_2024.csv",
    "fluctuation_2024": DATA_DIR / "fluctuation_2025.csv",
    "fluctuation_2025": DATA_DIR / "fluctuation_2026.csv",
    "median_income_2023": DATA_DIR / "median_income_2024.csv",
    "households_2020": DATA_DIR / "households_2021.csv",
    "households_2021": DATA_DIR / "households_2022.csv",
    "households_2022": DATA_DIR / "households_2023.csv",
    "households_2023": DATA_DIR / "households_2024.csv",
    "households_2024": DATA_DIR / "households_2025.csv",
    "mss_index_2020": DATA_DIR / "mss_index_2021.csv",
    "mss_index_2022": DATA_DIR / "mss_index_2023.csv",
    "mss_index_2024": DATA_DIR / "mss_index_2025.csv",
    "mss_context_2020": DATA_DIR / "mss_context_2021.csv",
    "mss_context_2022": DATA_DIR / "mss_context_2023.csv"
}

In [5]:
# check whether all files exist

for name, path in files.items():
    print(name, path.exists())

population_2020 True
population_2021 True
population_2022 True
population_2023 True
population_2024 True
population_2025 True
fluctuation_2020 True
fluctuation_2021 True
fluctuation_2022 True
fluctuation_2023 True
fluctuation_2024 True
fluctuation_2025 True
median_income_2023 True
households_2020 True
households_2021 True
households_2022 True
households_2023 True
households_2024 True
mss_index_2020 True
mss_index_2022 True
mss_index_2024 True
mss_context_2020 True
mss_context_2022 True


In [6]:
# before we use a function, let's check with one example, if the files are read correctly =)

df_check = pd.read_csv(files["population_2020"], delimiter=";")
df_check.head()

,plr_key,plr_name,res_count,age_1,age_2,age_3,age_4,age_5,age_6,age_7,...,age_2_migr_backgr,age_3_migr_backgr,age_4_migr_backgr,age_5_migr_backgr,age_6_migr_backgr,age_7_migr_backgr,age_8_migr_backgr,female_migr_backgr,EU_migr_backgr_count,date
0,1100101,Stülerstraße,3 419,156,143,28,442,1 162,360,377,...,118,25,333,851,214,167,212,982,540,31.12.2020
1,1100102,Großer Tiergarten,1 791,98,94,47,156,671,260,240,...,88,38,122,532,169,102,63,550,329,31.12.2020
2,1100103,Lützowstraße,5 211,264,306,102,604,1 578,638,740,...,248,84,382,1 044,368,335,369,1 449,890,31.12.2020
3,1100104,Körnerstraße,4 636,256,410,112,779,1 629,584,500,...,336,96,505,1 072,342,261,192,1 359,593,31.12.2020
4,1100205,Wilhelmstraße,2 573,109,139,51,316,827,338,349,...,121,39,225,629,209,125,149,746,385,31.12.2020


In [7]:
# checking, if column names are correct

df_check.columns

Index(['plr_key', 'plr_name', 'res_count', 'age_1', 'age_2', 'age_3', 'age_4',
       'age_5', 'age_6', 'age_7', 'age_8', 'female', 'non_ger_cit_count',
       'migr_bckgr_count', 'age_1_migr_backgr', 'age_2_migr_backgr',
       'age_3_migr_backgr', 'age_4_migr_backgr', 'age_5_migr_backgr',
       'age_6_migr_backgr', 'age_7_migr_backgr', 'age_8_migr_backgr',
       'female_migr_backgr', 'EU_migr_backgr_count', 'date'],
      dtype='object')

In [8]:
# checking for data types

df_check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542 entries, 0 to 541
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   plr_key               542 non-null    int64 
 1   plr_name              542 non-null    object
 2   res_count             542 non-null    object
 3   age_1                 542 non-null    object
 4   age_2                 542 non-null    object
 5   age_3                 542 non-null    object
 6   age_4                 542 non-null    object
 7   age_5                 542 non-null    object
 8   age_6                 542 non-null    object
 9   age_7                 542 non-null    object
 10  age_8                 542 non-null    object
 11  female                542 non-null    object
 12  non_ger_cit_count     542 non-null    object
 13  migr_bckgr_count      542 non-null    object
 14  age_1_migr_backgr     542 non-null    object
 15  age_2_migr_backgr     542 non-null    ob

In [9]:
df_check.dtypes

plr_key                  int64
plr_name                object
res_count               object
age_1                   object
age_2                   object
age_3                   object
age_4                   object
age_5                   object
age_6                   object
age_7                   object
age_8                   object
female                  object
non_ger_cit_count       object
migr_bckgr_count        object
age_1_migr_backgr       object
age_2_migr_backgr       object
age_3_migr_backgr       object
age_4_migr_backgr       object
age_5_migr_backgr       object
age_6_migr_backgr       object
age_7_migr_backgr       object
age_8_migr_backgr       object
female_migr_backgr      object
EU_migr_backgr_count    object
date                    object
dtype: object

In [10]:
# looking at specific columns to see how we need to clean the data 

df_check[["plr_key", "res_count", "age_1", "date"]].head(10)

# we can see that plr_key does NOT always have 8 digits, also numbers > 1000 are displayed as "1 000" and the "date" column does not have the data type datetime, but is displayed correctly

,plr_key,res_count,age_1,date
0,1100101,3 419,156,31.12.2020
1,1100102,1 791,98,31.12.2020
2,1100103,5 211,264,31.12.2020
3,1100104,4 636,256,31.12.2020
4,1100205,2 573,109,31.12.2020
5,1100206,1 803,108,31.12.2020
6,1100207,8 312,472,31.12.2020
7,1100308,6 967,446,31.12.2020
8,1100309,12 352,694,31.12.2020
9,1100310,9 903,474,31.12.2020


In [11]:
# let's try a couple of fixes, see if they work, before we build a whole function for cleaning each file individually

# first, we check if we can fix the plr_key length; it's supposed to have 8 digits, and if it doesn't have 8 digits, we need to add a "0" at the beginning

# we can convert the plr_key column into a string while reading the file:

df_check2 = pd.read_csv(files["population_2020"], delimiter = ";", dtype={"plr_key" : "string"})

In [12]:
df_check2.dtypes

plr_key                 string[python]
plr_name                        object
res_count                       object
age_1                           object
age_2                           object
age_3                           object
age_4                           object
age_5                           object
age_6                           object
age_7                           object
age_8                           object
female                          object
non_ger_cit_count               object
migr_bckgr_count                object
age_1_migr_backgr               object
age_2_migr_backgr               object
age_3_migr_backgr               object
age_4_migr_backgr               object
age_5_migr_backgr               object
age_6_migr_backgr               object
age_7_migr_backgr               object
age_8_migr_backgr               object
female_migr_backgr              object
EU_migr_backgr_count            object
date                            object
dtype: object

In [13]:
# and now fill the string with a "0" at the beginning

df_check2["plr_key"] = df_check2["plr_key"].str.zfill(8)
df_check2["plr_key"].head(10)

# seems to have worked =D

0    01100101
1    01100102
2    01100103
3    01100104
4    01100205
5    01100206
6    01100207
7    01100308
8    01100309
9    01100310
Name: plr_key, dtype: string

In [14]:
# check if all values have length 8 now

df_check2["plr_key"].str.len().value_counts()

# yay! so we can fix this using the function above =)

plr_key
8    542
Name: count, dtype: Int64

In [15]:
# next try: fix "date" column: convert into datetime!

df_check2["date"] = pd.to_datetime(df_check2["date"], dayfirst=True)
df_check2["date"].dtype

dtype('<M8[ns]')

In [16]:
df_check2["date"].head(9)

# WORKED! so we can also use this for our cleaning function =)

0   2020-12-31
1   2020-12-31
2   2020-12-31
3   2020-12-31
4   2020-12-31
5   2020-12-31
6   2020-12-31
7   2020-12-31
8   2020-12-31
Name: date, dtype: datetime64[ns]

In [17]:
# now let's try clean numbers that are > 1000 but displayed with a space between

# pandas reads them as objects, so we can treat them accordingly =D

df_check2["res_count"] = df_check2["res_count"].str.strip().str.replace(" ", "", regex=False)

In [18]:
# now convert object into integer

# before we can do that, we need to replace "-" in the column with nan (these are missing values)

df_check2["res_count"] = df_check2["res_count"].replace(["–"], np.nan)

# and then convert

df_check2["res_count"] = pd.to_numeric(df_check2["res_count"])

# works =D

In [19]:
# now let's look at the other files, see if we need to pay attention to something else as well:

df_check3 = pd.read_csv(files["fluctuation_2020"], delimiter = ";", dtype={"plr_key" : "string"})

In [20]:
df_check3.head(20)

,plr_key,ext_in_mig_total_count,ext_in_migr_under_18_count,ext_in_migr_18_to_under_27_count,ext_in_migr_27_to_under_45_count,ext_in_migr_45_to_under_65_count,ext_in_migr_65_to_under_80_count,ext_in_migr_80_and_over_count,ext_out_migr_total_count,ext_out_migr_under_18_count,...,int_in_migr_65_to_under_80_count,int_in_migr_80_and_over_count,int_out_migr_total_count,int_out_migr_under_18_count,int_out_migr_18_to_under_27_count,int_out_migration_27_to_under_45_count,int_out_migr_45_to_under_65_count,int_out_migr_65_to_under_80_count,int_out_migr_80_and_over_count,date
0,1100101,267,21,100,118,13,15,0,222,33,...,9,6,345,47,94,154,31,16,3,31.12.2020
1,1100102,218,31,41,113,21,9,3,175,15,...,0,0,186,36,15,94,35,0,6,31.12.2020
2,1100103,359,25,119,148,58,9,0,354,24,...,16,15,377,55,78,178,40,23,3,31.12.2020
3,1100104,367,24,150,160,30,3,0,363,24,...,9,0,443,32,126,249,30,6,0,31.12.2020
4,1100205,415,55,123,190,44,0,3,289,34,...,0,0,278,68,49,121,27,9,4,31.12.2020
5,1100206,170,10,29,101,30,0,0,180,13,...,0,3,162,37,21,79,18,7,0,31.12.2020
6,1100207,596,48,150,314,71,10,3,546,45,...,12,3,750,98,119,427,79,16,11,31.12.2020
7,1100308,699,56,168,398,65,9,3,518,59,...,9,6,755,101,116,449,80,9,0,31.12.2020
8,1100309,796,76,199,438,80,3,0,766,103,...,6,0,1071,145,156,599,140,25,6,31.12.2020
9,1100310,1015,93,279,571,63,9,0,676,38,...,3,0,1183,123,224,735,73,15,13,31.12.2020


In [21]:
df_check3.dtypes

plr_key                                   string[python]
ext_in_mig_total_count                             int64
ext_in_migr_under_18_count                         int64
ext_in_migr_18_to_under_27_count                   int64
ext_in_migr_27_to_under_45_count                   int64
ext_in_migr_45_to_under_65_count                   int64
ext_in_migr_65_to_under_80_count                   int64
ext_in_migr_80_and_over_count                      int64
ext_out_migr_total_count                           int64
ext_out_migr_under_18_count                        int64
ext_out_migr_18_to_under_27_count                  int64
ext_out_migr_27_to_under_45_count                  int64
ext_out_migr_45_to_under_65_count                  int64
ext_out_migr_65_to_under_80_count                  int64
ext_out_migr_80_and_over_count                     int64
int_in_migr_total_count                            int64
int_in_migr_under_18_count                         int64
int_in_migr_18_to_under_27_coun

In [22]:
# we'll check if there are symbols for missing data as well; dtypes gives us a hint at "there should be none", since otherwise they would be read as "object" columns, not as "int", but let's still check :P

df_check3.astype("string").isin(["*", "-", "–", ""]).sum()

# nothing to see here, so let's continue with the next file :P

plr_key                                   0
ext_in_mig_total_count                    0
ext_in_migr_under_18_count                0
ext_in_migr_18_to_under_27_count          0
ext_in_migr_27_to_under_45_count          0
ext_in_migr_45_to_under_65_count          0
ext_in_migr_65_to_under_80_count          0
ext_in_migr_80_and_over_count             0
ext_out_migr_total_count                  0
ext_out_migr_under_18_count               0
ext_out_migr_18_to_under_27_count         0
ext_out_migr_27_to_under_45_count         0
ext_out_migr_45_to_under_65_count         0
ext_out_migr_65_to_under_80_count         0
ext_out_migr_80_and_over_count            0
int_in_migr_total_count                   0
int_in_migr_under_18_count                0
int_in_migr_18_to_under_27_count          0
int_in_migr_27_to_under_45_count          0
int_in_migr_45_to_under_65_count          0
int_in_migr_65_to_under_80_count          0
int_in_migr_80_and_over_count             0
int_out_migr_total_count        

In [23]:
# let's check the median_income file

df_check4 = pd.read_csv(files["median_income_2023"], delimiter = ";", dtype={"plr_key" : "string"})

In [24]:
df_check4.head()

# we can see that the percentage has a comma instead of a period, we need to keep that in mind and fix it

,plr_key,plr_name,ft_employee_ss_count,median_income_eur,median_wage_quintile,lw_ft_employee_ss_count,lw_ft_employee_ss_perc,date
0,1100101,Stülerstraße,866,4721,1.0,96,"11,1",31.12.2023
1,1100102,Großer Tiergarten,619,6279,1.0,33,"5,3",31.12.2023
2,1100103,Lützowstraße,1250,4895,1.0,144,"11,5",31.12.2023
3,1100104,Körnerstraße,1126,5062,1.0,134,"11,9",31.12.2023
4,1100205,Wilhelmstraße,828,5384,1.0,111,"13,4",31.12.2023


In [25]:
df_check4.dtypes

plr_key                    string[python]
plr_name                           object
ft_employee_ss_count               object
median_income_eur                  object
median_wage_quintile              float64
lw_ft_employee_ss_count            object
lw_ft_employee_ss_perc             object
date                               object
dtype: object

In [26]:
df_check4.astype("string").isin(["*", "-", "–", "", "."]).sum()

# we also have symbols for missing values in here!

plr_key                     0
plr_name                    0
ft_employee_ss_count       14
median_income_eur           0
median_wage_quintile        0
lw_ft_employee_ss_count    15
lw_ft_employee_ss_perc      1
date                        0
dtype: int64

In [27]:
# let's try and fix the comma issue:

df_check4["lw_ft_employee_ss_perc"] = df_check4["lw_ft_employee_ss_perc"].str.replace(",", ".", regex=False)

df_check4.head()

# worked =)

,plr_key,plr_name,ft_employee_ss_count,median_income_eur,median_wage_quintile,lw_ft_employee_ss_count,lw_ft_employee_ss_perc,date
0,1100101,Stülerstraße,866,4721,1.0,96,11.1,31.12.2023
1,1100102,Großer Tiergarten,619,6279,1.0,33,5.3,31.12.2023
2,1100103,Lützowstraße,1250,4895,1.0,144,11.5,31.12.2023
3,1100104,Körnerstraße,1126,5062,1.0,134,11.9,31.12.2023
4,1100205,Wilhelmstraße,828,5384,1.0,111,13.4,31.12.2023


In [28]:
# let's continue with the household files:

df_check5 = pd.read_csv(files["households_2022"], delimiter = ";", dtype={"plr_key" : "string"})

df_check5


,plr_key,plr_name,res_count,single_pers_hh_count,pers_multi_pers_hh_count,pers_two_pers_hh_count,pers_three_pers_hh_count,pers_four_or_more_pers_hh_count,avg_hh_size,households_total,...,single_pers_hh_30_to_under_65_count,single_pers_hh_65_and_over_count,single_pers_hh_female_count,hh_wo_child_count,hh_with_child_total_count,hh_with_one_child_count,hh_with_two_child_count,hh_with_three_or_more_child_count,single_parent_hh_share_perc,date
0,1100101,Stülerstraße,3 698,1 421,2 277,1 128,501,648,"1,61",2 295,...,748,418,628,2 062,233,126,78,29,"26,6",31.12.2022
1,1100102,Großer Tiergarten,2 166,623,1 543,714,372,457,"1,8",1 206,...,376,126,240,1 026,180,112,57,11,"26,1",31.12.2022
2,1100103,Lützowstraße,5 545,1 931,3 614,1 586,750,1 278,"1,71",3 252,...,1 052,431,823,2 817,435,235,136,64,28,31.12.2022
3,1100104,Körnerstraße,5 043,1 544,3 499,1 348,717,1 434,"1,82",2 764,...,930,180,564,2 320,444,220,165,59,"25,2",31.12.2022
4,1100205,Wilhelmstraße,3 098,889,2 209,996,519,694,"1,81",1 714,...,454,183,364,1 482,232,123,78,31,"19,4",31.12.2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,12601032,Rollbergesiedlung,6 091,1 185,4 906,1 366,1 110,2 430,"2,22",2 747,...,584,440,542,1 985,762,280,269,213,"34,9",31.12.2022
538,12601133,Treuenbrietzener Straße,11 308,2 041,9 267,2 696,2 031,4 540,"2,25",5 032,...,1 019,787,1 017,3 617,1 415,585,510,320,"36,9",31.12.2022
539,12601134,Märkisches Zentrum,15 581,2 894,12 687,4 214,2 685,5 788,"2,19",7 101,...,1 354,1 206,1 531,5 294,1 807,766,608,433,"32,7",31.12.2022
540,12601235,Dannenwalder Weg,10 724,1 791,8 933,2 374,1 791,4 768,"2,35",4 565,...,856,679,886,3 250,1 315,500,440,375,"31,9",31.12.2022


In [29]:
# here we definitely have the same "comma instead of period" problem, as well as the "space" problem with numbers larger than 999. Let's check for symbols for missing values

df_check5.astype("string").isin(["*", "-", "–", "", "."]).sum()

# yupp. lots.

plr_key                                      0
plr_name                                     0
res_count                                    1
single_pers_hh_count                         1
pers_multi_pers_hh_count                     2
pers_two_pers_hh_count                       2
pers_three_pers_hh_count                     3
pers_four_or_more_pers_hh_count              3
avg_hh_size                                  0
households_total                             1
multi_pers_hh_tot_count                      2
two_pers_hh_count                            2
three_pers_hh_count                          3
four_or_more_pers_hh_count                   3
pers_married_couple_hh_tot_count             2
pers_married_couple_hh_wo_child_count        2
pers_married_couple_hh_with_child_count      3
pers_unmarried_couple_hh_tot_count           3
pers_unmarried_couple_hh_wo_child_count      3
pers_unmarried_couple_hh_with_child_count    3
pers_single_parent_hh_count                  2
pers_other_mu

In [30]:
# let's check mss_index now:

df_check6 = pd.read_csv(files["mss_index_2022"], delimiter = ";", dtype={"plr_key" : "string"})

df_check6.head()

# we have the same decimal comma issue and also the numbers should be rounded! let's keep that in mind for our cleaning function

,plr_key,plr_name,res_count,s1_unempl_status_indicator,s2_single_parent_child_status_indicator,s3_transfer_benefit_status_indicator,s4_child_poverty_status_indicator,d1_unempl_dynamics_indicator,d2_single_parent_child_dynamics_indicator,d3_transfer_benefit_dynamics_indicator,d4_child_poverty_dynamics_indicator,date
0,1100101,Stülerstraße,3561,"2,7589","23,5589","8,70542","19,51872","-1,46704","-0,29431","0,66213","0,12072",31.12.2022
1,1100102,Großer Tiergarten,2002,"1,1889","21,93309","3,3966","7,85124","0,03032","-0,66106","-0,06515","-0,48209",31.12.2022
2,1100103,Lützowstraße,5662,"3,7308","23,52113","13,22854","27,79605","-1,58464","0,60446","-2,29631","-4,83553",31.12.2022
3,1100104,Körnerstraße,5029,"4,89266","20,40302","15,11235","31,50685","-1,28835","-0,29106","-2,89887","-8,43309",31.12.2022
4,1100205,Wilhelmstraße,3009,"1,60571","20,95491","5,88235","13,23077","-1,27689","0,21912","-0,45266","-6,5273",31.12.2022


In [31]:
# let's check mss_context files:

df_check7 = pd.read_csv(files["mss_context_2022"], delimiter = ";", dtype={"plr_key" : "string"})

df_check7.head()

# same issues as with the index file, in addition we have completely empty columns

,plr_key,plr_name,res_count,k1_youth_unempl_sgb2_share_pct,k2_single_parent_hh_with_children_share_pct,k3_old_age_basic_income_support_share_pct,k18_fem_old_age_basic_income_support_share_pct,k19_male_old_age_basic_income_support_share_pct,k4_child_adol_migr_bckgr_share_pct,k5_res_migr_bckgr_share_pct,...,k7_foreign_sgb2_transfer_recipients_share_pct,k8_municipal_housing_units_share_pct,k14_avg_rooms_per_res,k15_avg_living_space_sqm_per_res,k9_res_simple_resid_loc_share_pct,k10_res_same_addr_5plus_years_share_pct,k11_migr_vol_2yr_avg_annual_pct,k12_net_migr_total_2yr_avg_annual_pct,k13_net_migr_under_6_2yr_avg_annual_pct,date
0,01100101,Stülerstraße,3561,"2,24719","26,60944","10,78717","13,11475","8,125","85,71429","64,08312",...,"9,44717",NaN,NaN,NaN,0,"52,64234","37,77722","2,1435","4,93908",31.12.2022
1,01100102,Großer Tiergarten,2002,0,"26,11111","6,91057",0,0,"88,10409","70,82917",...,"4,00391",NaN,NaN,NaN,"0,84915","37,61131","52,58348","5,43093","10,70153",31.12.2022
2,01100103,Lützowstraße,5662,"2,82392","28,04598","19,83211","21,37405","17,94872","82,39437","61,32109",...,"15,94937",NaN,NaN,NaN,"8,86613","56,08856","32,08897","5,4887","-1,33141",31.12.2022
3,01100104,Körnerstraße,5029,"3,31825","25,22523","30,32787","30,21978","30,43478","80,10076","66,87214",...,"17,90895",NaN,NaN,NaN,"6,9994","54,19355","36,14178","3,63736","-1,12777",31.12.2022
4,01100205,Wilhelmstraße,3009,0,"19,39655","6,56109",0,0,"88,85942","67,99601",...,"4,9458",NaN,NaN,NaN,0,"48,72061","65,21981","8,40415","8,32627",31.12.2022


In [32]:
# OK, let's write the cleaning function- first, for each file individually:

def clean_single_table(file_path):
    df = pd.read_csv(
        file_path,
        delimiter=";",
        dtype={"plr_key": "string"}
    )

    # clean column names
    df.columns = (
        df.columns
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    # clean plr_key and date columns
    df["plr_key"] = df["plr_key"].str.zfill(8)
    df["date"] = pd.to_datetime(df["date"], dayfirst=True)

    # define columns that should not be converted to numeric (to define numeric columns)
    non_num_cols = ["plr_key", "plr_name", "date"]

    num_cols = [
        col for col in df.columns
        if col not in non_num_cols
    ]

    # clean and convert numeric columns
    for col in num_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace(["*", "-", "–", "", ".", "•"], np.nan)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
        )

        df[col] = pd.to_numeric(df[col])

    # round float columns
    float_cols = df.select_dtypes(include="float").columns
    df[float_cols] = df[float_cols].round(3)

    return df

In [33]:
# let's test the function =D

test_clean = clean_single_table(files["households_2023"])

test_clean.head()

# looks pretty good =D

,plr_key,plr_name,res_count,single_pers_hh_count,pers_multi_pers_hh_count,pers_two_pers_hh_count,pers_three_pers_hh_count,pers_four_or_more_pers_hh_count,avg_hh_size,households_total,...,single_pers_hh_30_to_under_65_count,single_pers_hh_65_and_over_count,single_pers_hh_female_count,hh_wo_child_count,hh_with_child_total_count,hh_with_one_child_count,hh_with_two_child_count,hh_with_three_or_more_child_count,single_parent_hh_share_perc,date
0,01100101,Stülerstraße,3905,1535,2370,1258,516,596,1.58,2468,...,788,439,688,2216,252,146,78,28,25.8,2023-12-31
1,01100102,Großer Tiergarten,2203,696,1507,758,375,374,1.71,1285,...,433,138,256,1109,176,114,52,10,23.3,2023-12-31
2,01100103,Lützowstraße,5704,2129,3575,1620,747,1208,1.65,3457,...,1189,496,900,3006,451,241,150,60,27.9,2023-12-31
3,01100104,Körnerstraße,5110,1674,3436,1280,771,1385,1.78,2865,...,1017,198,659,2423,442,224,158,60,19.9,2023-12-31
4,01100205,Wilhelmstraße,3095,1003,2092,1036,462,594,1.71,1809,...,525,192,412,1589,220,116,81,23,17.3,2023-12-31


In [34]:
test_clean.dtypes

plr_key                                      string[python]
plr_name                                             object
res_count                                             Int64
single_pers_hh_count                                  Int64
pers_multi_pers_hh_count                              Int64
pers_two_pers_hh_count                                Int64
pers_three_pers_hh_count                              Int64
pers_four_or_more_pers_hh_count                       Int64
avg_hh_size                                         Float64
households_total                                      Int64
multi_pers_hh_tot_count                               Int64
two_pers_hh_count                                     Int64
three_pers_hh_count                                   Int64
four_or_more_pers_hh_count                            Int64
pers_married_couple_hh_tot_count                      Int64
pers_married_couple_hh_wo_child_count                 Int64
pers_married_couple_hh_with_child_count 

In [35]:
# let's go through all tables now =)

# we already have the dictionary "files", and can access both, keys and values, with .items(), so we can use that here and save the result in a new dictionary "cleaned_dfs":

cleaned_dfs = {}

for name, path in files.items():
    cleaned_dfs[name] = clean_single_table(path)
    print(f"{cleaned_dfs[name].shape}") # prints names and shapes of cleaned dfs to check =)

(542, 25)
(542, 25)
(542, 25)
(542, 25)
(542, 25)
(542, 25)
(542, 30)
(542, 30)
(542, 30)
(542, 30)
(542, 30)
(542, 30)
(542, 8)
(542, 41)
(542, 41)
(542, 41)
(542, 41)
(542, 41)
(542, 12)
(542, 12)
(542, 12)
(542, 23)
(542, 23)


In [36]:
# let's do a quick quality check:

for name, df in cleaned_dfs.items():
    print(name)
    print("shape:", df.shape)
    print("plr_key lengths:")
    print(df["plr_key"].str.len().value_counts())   # checks, if the plr_keys are right now (= have 8 digits)
    print("duplicates plr_key + date:", df.duplicated(subset=["plr_key", "date"]).sum())    # checks for duplicates in plr_keys plus date
    print()

population_2020
shape: (542, 25)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

population_2021
shape: (542, 25)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

population_2022
shape: (542, 25)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

population_2023
shape: (542, 25)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

population_2024
shape: (542, 25)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

population_2025
shape: (542, 25)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

fluctuation_2020
shape: (542, 30)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

fluctuation_2021
shape: (542, 30)
plr_key lengths:
plr_key
8    542
Name: count, dtype: Int64
duplicates plr_key + date: 0

fluctuation_20

In [37]:
# and now: a quality report for all cleaned tables, cause we looove quality :P

quality_rows = []

for name, df in cleaned_dfs.items():
    n_rows = df.shape[0]
    n_cols = df.shape[1]

    duplicate_key_date_count = df.duplicated(subset=["plr_key", "date"]).sum()

    plr_key_min_length = df["plr_key"].str.len().min()
    plr_key_max_length = df["plr_key"].str.len().max()

    missing_total = df.isna().sum().sum()
    missing_share = df.isna().sum().sum() / df.size

    empty_columns = df.columns[df.isna().all()].tolist()

    quality_rows.append({
        "table_name": name,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "expected_542_rows": n_rows == 542,
        "duplicate_plr_key_date_count": duplicate_key_date_count,
        "plr_key_min_length": plr_key_min_length,
        "plr_key_max_length": plr_key_max_length,
        "plr_key_all_8_digits": (plr_key_min_length == 8) and (plr_key_max_length == 8),
        "missing_total": missing_total,
        "missing_share": round(missing_share, 3),
        "empty_columns_count": len(empty_columns),
        "empty_columns": empty_columns
    })

quality_report = pd.DataFrame(quality_rows)

quality_report

,table_name,n_rows,n_cols,expected_542_rows,duplicate_plr_key_date_count,plr_key_min_length,plr_key_max_length,plr_key_all_8_digits,missing_total,missing_share,empty_columns_count,empty_columns
0,population_2020,542,25,True,0,8,8,True,87,0.006,0,[]
1,population_2021,542,25,True,0,8,8,True,73,0.005,0,[]
2,population_2022,542,25,True,0,8,8,True,72,0.005,0,[]
3,population_2023,542,25,True,0,8,8,True,72,0.005,0,[]
4,population_2024,542,25,True,0,8,8,True,76,0.006,0,[]
5,population_2025,542,25,True,0,8,8,True,78,0.006,0,[]
6,fluctuation_2020,542,30,True,0,8,8,True,0,0.000,0,[]
7,fluctuation_2021,542,30,True,0,8,8,True,0,0.000,0,[]
8,fluctuation_2022,542,30,True,0,8,8,True,0,0.000,0,[]
9,fluctuation_2023,542,30,True,0,8,8,True,0,0.000,0,[]


In [38]:
# only show tables with potential quality issues:

quality_report[
    (quality_report["expected_542_rows"] == False)
    | (quality_report["duplicate_plr_key_date_count"] > 0)
    | (quality_report["plr_key_all_8_digits"] == False)
    | (quality_report["empty_columns_count"] > 0)
]

,table_name,n_rows,n_cols,expected_542_rows,duplicate_plr_key_date_count,plr_key_min_length,plr_key_max_length,plr_key_all_8_digits,missing_total,missing_share,empty_columns_count,empty_columns
13,households_2020,542,41,True,0,8,8,True,801,0.036,1,[hh_wo_child_count]
14,households_2021,542,41,True,0,8,8,True,635,0.029,1,[hh_wo_child_count]
18,mss_index_2020,542,12,True,0,8,8,True,1120,0.172,2,"[s2_single_parent_child_status_indicator, d2_s..."
21,mss_context_2020,542,23,True,0,8,8,True,2796,0.224,5,[k18_fem_old_age_basic_income_support_share_pc...
22,mss_context_2022,542,23,True,0,8,8,True,1722,0.138,3,"[k8_municipal_housing_units_share_pct, k14_avg..."


In [40]:
# let's further inspect those tables:

for table_name in ["mss_index_2020", "mss_context_2020", "mss_context_2022"]:
    print(f"\n{table_name}")
    missing_by_column = cleaned_dfs[table_name].isna().sum()
    display(missing_by_column[missing_by_column > 0].sort_values(ascending=False))


mss_index_2020


s2_single_parent_child_status_indicator      542
d2_single_parent_child_dynamics_indicator    542
s1_unempl_status_indicator                     6
s3_transfer_benefit_status_indicator           6
s4_child_poverty_status_indicator              6
d1_unempl_dynamics_indicator                   6
d3_transfer_benefit_dynamics_indicator         6
d4_child_poverty_dynamics_indicator            6
dtype: int64


mss_context_2020


k8_municipal_housing_units_share_pct               542
k18_fem_old_age_basic_income_support_share_pct     542
k19_male_old_age_basic_income_support_share_pct    542
k15_avg_living_space_sqm_per_res                   542
k14_avg_rooms_per_res                              542
k2_single_parent_hh_with_children_share_pct          8
k1_youth_unempl_sgb2_share_pct                       6
k12_net_migr_total_2yr_avg_annual_pct                6
k11_migr_vol_2yr_avg_annual_pct                      6
k10_res_same_addr_5plus_years_share_pct              6
k9_res_simple_resid_loc_share_pct                    6
k17_non_eu_foreign_res_share_pct                     6
k7_foreign_sgb2_transfer_recipients_share_pct        6
k6_foreign_res_share_change_2yr_pp                   6
k16_foreign_res_share_pct                            6
k5_res_migr_bckgr_share_pct                          6
k4_child_adol_migr_bckgr_share_pct                   6
k3_old_age_basic_income_support_share_pct            6
k13_net_mi


mss_context_2022


k15_avg_living_space_sqm_per_res                   542
k14_avg_rooms_per_res                              542
k8_municipal_housing_units_share_pct               542
k1_youth_unempl_sgb2_share_pct                       6
k7_foreign_sgb2_transfer_recipients_share_pct        6
k12_net_migr_total_2yr_avg_annual_pct                6
k11_migr_vol_2yr_avg_annual_pct                      6
k10_res_same_addr_5plus_years_share_pct              6
k9_res_simple_resid_loc_share_pct                    6
k17_non_eu_foreign_res_share_pct                     6
k2_single_parent_hh_with_children_share_pct          6
k6_foreign_res_share_change_2yr_pp                   6
k16_foreign_res_share_pct                            6
k5_res_migr_bckgr_share_pct                          6
k4_child_adol_migr_bckgr_share_pct                   6
k19_male_old_age_basic_income_support_share_pct      6
k18_fem_old_age_basic_income_support_share_pct       6
k3_old_age_basic_income_support_share_pct            6
k13_net_mi

Note:
Some MSS indicators are fully missing in earlier years because hey were not yet part of the MSS indicator system (mss context file) or not available in the published data (e.g. index file 2021, where S2/D2 - children in single parents households - are missing because of missing data in the years (covid?)).
We keep those columns for now and decide later to drop them or not.

In [41]:
# let's also check for data types after cleaning:

dtype_rows = []

for table_name, df in cleaned_dfs.items():
    for col in df.columns:
        dtype_rows.append({
            "table_name": table_name,
            "column_name": col,
            "dtype": str(df[col].dtype)
        })

dtype_report = pd.DataFrame(dtype_rows)

dtype_report

,table_name,column_name,dtype
0,population_2020,plr_key,string
1,population_2020,plr_name,object
2,population_2020,res_count,Int64
3,population_2020,age_1,Int64
4,population_2020,age_2,Int64
...,...,...,...
620,mss_context_2022,k10_res_same_addr_5plus_years_share_pct,Float64
621,mss_context_2022,k11_migr_vol_2yr_avg_annual_pct,Float64
622,mss_context_2022,k12_net_migr_total_2yr_avg_annual_pct,Float64
623,mss_context_2022,k13_net_migr_under_6_2yr_avg_annual_pct,Float64


In [42]:
# only show columns that are still object dtype after cleaning

dtype_report[dtype_report["dtype"] == "object"]

# only plr_name columns! perfect =D

,table_name,column_name,dtype
1,population_2020,plr_name,object
26,population_2021,plr_name,object
51,population_2022,plr_name,object
76,population_2023,plr_name,object
101,population_2024,plr_name,object
126,population_2025,plr_name,object
331,median_income_2023,plr_name,object
339,households_2020,plr_name,object
380,households_2021,plr_name,object
421,households_2022,plr_name,object


In [44]:
# now let's bring the data tables together for each "topic":

population_all = pd.concat(
    [
        cleaned_dfs["population_2020"],
        cleaned_dfs["population_2021"],
        cleaned_dfs["population_2022"],
        cleaned_dfs["population_2023"],
        cleaned_dfs["population_2024"],
        cleaned_dfs["population_2025"],
    ],
    ignore_index=True
)

fluctuation_all = pd.concat(
    [
        cleaned_dfs["fluctuation_2020"],
        cleaned_dfs["fluctuation_2021"],
        cleaned_dfs["fluctuation_2022"],
        cleaned_dfs["fluctuation_2023"],
        cleaned_dfs["fluctuation_2024"],
        cleaned_dfs["fluctuation_2025"]
    ],
    ignore_index=True
)

households_all = pd.concat(
    [
        cleaned_dfs["households_2020"],
        cleaned_dfs["households_2021"],
        cleaned_dfs["households_2022"],
        cleaned_dfs["households_2023"],
        cleaned_dfs["households_2024"],
    ],
    ignore_index=True
)

mss_index_all = pd.concat(
    [
        cleaned_dfs["mss_index_2020"],
        cleaned_dfs["mss_index_2022"],
        cleaned_dfs["mss_index_2024"],
    ],
    ignore_index=True
)

mss_context_all = pd.concat(
    [
        cleaned_dfs["mss_context_2020"],
        cleaned_dfs["mss_context_2022"],
    ],
    ignore_index=True
)

median_income_all = cleaned_dfs["median_income_2023"].copy()

In [45]:
# let's check what we did:

print("population_all:", population_all.shape)
print("fluctuation_all:", fluctuation_all.shape)
print("households_all:", households_all.shape)
print("mss_index_all:", mss_index_all.shape)
print("mss_context_all:", mss_context_all.shape)
print("median_income_all:", median_income_all.shape)

# seems to be right =)

population_all: (3252, 25)
fluctuation_all: (3252, 30)
households_all: (2710, 41)
mss_index_all: (1626, 12)
mss_context_all: (1084, 23)
median_income_all: (542, 8)


Now onto the next problem before we do the final merge: the population files and the household files both contain a column for resident count in the PLR. But the numbers don't match.
According to the metadata, the household count is based on the population registered with main or secondary residence and excludes persons without own household formation, such as persons in institutions.
With the following code we will explore how much the counts differ, but we will decide to keep only the resident count from the population file, as this is more precise, counting only population registered with main residence.

In [46]:
# Compare res_count across topic tables before dropping duplicate res_count columns

res_count_checks = []

tables_with_res_count = {
    "households": households_all,
    "mss_index": mss_index_all,
    "mss_context": mss_context_all,
}

for table_name, table in tables_with_res_count.items():
    compare_df = population_all[["plr_key", "date", "res_count"]].merge(
        table[["plr_key", "date", "res_count"]],
        on=["plr_key", "date"],
        how="inner",
        suffixes=("_population", f"_{table_name}")
    )

    compare_df["res_count_difference"] = (
        compare_df[f"res_count_{table_name}"] - compare_df["res_count_population"]
    )

    res_count_checks.append({
        "table_name": table_name,
        "n_compared_rows": len(compare_df),
        "n_differences": (compare_df["res_count_difference"] != 0).sum(),
        "max_absolute_difference": compare_df["res_count_difference"].abs().max()
    })

res_count_report = pd.DataFrame(res_count_checks)

res_count_report

,table_name,n_compared_rows,n_differences,max_absolute_difference
0,households,2710,2692,2161
1,mss_index,1626,0,0
2,mss_context,1084,0,0


In [47]:
# Compare household res_count with population res_count across all possible dates (just to exclude the possibility of having matched them to the wrong date)

date_match_rows = []

population_dates = sorted(population_all["date"].unique())
household_dates = sorted(households_all["date"].unique())

for household_date in household_dates:
    household_tmp = households_all.loc[
        households_all["date"] == household_date,
        ["plr_key", "res_count"]
    ]

    for population_date in population_dates:
        population_tmp = population_all.loc[
            population_all["date"] == population_date,
            ["plr_key", "res_count"]
        ]

        compare_tmp = household_tmp.merge(
            population_tmp,
            on="plr_key",
            how="inner",
            suffixes=("_households", "_population")
        )

        compare_tmp["difference"] = (
            compare_tmp["res_count_households"]
            - compare_tmp["res_count_population"]
        )

        date_match_rows.append({
            "household_date": household_date,
            "population_date": population_date,
            "n_compared_rows": len(compare_tmp),
            "n_differences": (compare_tmp["difference"] != 0).sum(),
            "mean_abs_difference": compare_tmp["difference"].abs().mean(),
            "max_abs_difference": compare_tmp["difference"].abs().max(),
            "total_abs_difference": compare_tmp["difference"].abs().sum()
        })

date_match_report = pd.DataFrame(date_match_rows)

date_match_report.sort_values(
    ["household_date", "total_abs_difference"]
)

,household_date,population_date,n_compared_rows,n_differences,mean_abs_difference,max_abs_difference,total_abs_difference
0,2020-12-31,2020-12-31,542,533,156.699065,1127,83834
1,2020-12-31,2021-12-31,542,534,213.755140,1493,114359
2,2020-12-31,2022-12-31,542,535,250.685981,2028,134117
3,2020-12-31,2023-12-31,542,534,304.833645,2537,163086
4,2020-12-31,2024-12-31,542,534,359.484112,3998,192324
5,2020-12-31,2025-12-31,542,535,403.108411,2866,215663
7,2021-12-31,2021-12-31,542,541,157.277264,1033,85087
6,2021-12-31,2020-12-31,542,538,165.324074,1050,89275
8,2021-12-31,2022-12-31,542,539,207.307407,1935,111946
9,2021-12-31,2023-12-31,542,539,268.124074,2075,144787


In [48]:
best_date_matches = (
    date_match_report
    .sort_values("total_abs_difference")
    .groupby("household_date")
    .head(1)
    .sort_values("household_date")
)

best_date_matches

# we see that the dates seem to be right. We will proceed with dropping the res_count column from the household files and only keep the res_count ffrom the population files.

,household_date,population_date,n_compared_rows,n_differences,mean_abs_difference,max_abs_difference,total_abs_difference
0,2020-12-31,2020-12-31,542,533,156.699065,1127,83834
7,2021-12-31,2021-12-31,542,541,157.277264,1033,85087
14,2022-12-31,2022-12-31,542,540,167.618519,1625,90514
21,2023-12-31,2023-12-31,542,540,178.396296,1246,96334
28,2024-12-31,2024-12-31,542,538,178.431481,2161,96353


In [49]:
# Ok, onto the final merge now =)

merge_keys = ["plr_key", "date"]

final_df = population_all.copy()

tables_to_merge = [
    fluctuation_all,
    households_all,
    median_income_all,
    mss_index_all,
    mss_context_all,
]

# since we have plr_name and also res_count more than once (see above), we will drop those

for table in tables_to_merge:
    table = table.drop(columns=["plr_name", "res_count"], errors="ignore")
    final_df = final_df.merge(
        table,
        on=merge_keys,
        how="outer"
    )

final_df.shape

(3252, 122)

In [50]:
# let's look at our result xD

final_df.head()

,plr_key,plr_name,res_count,age_1,age_2,age_3,age_4,age_5,age_6,age_7,...,k17_non_eu_foreign_res_share_pct,k7_foreign_sgb2_transfer_recipients_share_pct,k8_municipal_housing_units_share_pct,k14_avg_rooms_per_res,k15_avg_living_space_sqm_per_res,k9_res_simple_resid_loc_share_pct,k10_res_same_addr_5plus_years_share_pct,k11_migr_vol_2yr_avg_annual_pct,k12_net_migr_total_2yr_avg_annual_pct,k13_net_migr_under_6_2yr_avg_annual_pct
0,01100101,Stülerstraße,3419,156,143,28,442,1162,360,377,...,70.44,8.554,<NA>,<NA>,<NA>,0.0,55.917,38.361,0.507,-4.83
1,01100102,Großer Tiergarten,1791,98,94,47,156,671,260,240,...,71.287,3.079,<NA>,<NA>,<NA>,0.614,40.152,49.558,5.179,1.509
2,01100103,Lützowstraße,5211,264,306,102,604,1578,638,740,...,65.839,21.037,<NA>,<NA>,<NA>,6.045,57.487,29.779,2.41,0.295
3,01100104,Körnerstraße,4636,256,410,112,779,1629,584,500,...,75.937,25.295,<NA>,<NA>,<NA>,8.24,59.032,32.693,-2.087,-2.648
4,01100205,Wilhelmstraße,2573,109,139,51,316,827,338,349,...,74.25,7.048,<NA>,<NA>,<NA>,0.0,56.436,42.221,3.043,-5.98


In [51]:
final_df[["plr_key", "plr_name", "date"]].head()

,plr_key,plr_name,date
0,01100101,Stülerstraße,2020-12-31
1,01100102,Großer Tiergarten,2020-12-31
2,01100103,Lützowstraße,2020-12-31
3,01100104,Körnerstraße,2020-12-31
4,01100205,Wilhelmstraße,2020-12-31


In [52]:
final_df.duplicated(subset=["plr_key", "date"]).sum()

0

In [53]:
final_df["date"].value_counts().sort_index()

date
2020-12-31    542
2021-12-31    542
2022-12-31    542
2023-12-31    542
2024-12-31    542
2025-12-31    542
Name: count, dtype: int64

In [54]:
# Now let's clean up a bit: if there are completely empty columns, we don't need to keep them

empty_columns = final_df.columns[final_df.isna().all()].tolist()
empty_columns

['k8_municipal_housing_units_share_pct',
 'k14_avg_rooms_per_res',
 'k15_avg_living_space_sqm_per_res']

In [55]:
final_df = final_df.drop(columns=empty_columns)
final_df.head()

,plr_key,plr_name,res_count,age_1,age_2,age_3,age_4,age_5,age_6,age_7,...,k5_res_migr_bckgr_share_pct,k16_foreign_res_share_pct,k6_foreign_res_share_change_2yr_pp,k17_non_eu_foreign_res_share_pct,k7_foreign_sgb2_transfer_recipients_share_pct,k9_res_simple_resid_loc_share_pct,k10_res_same_addr_5plus_years_share_pct,k11_migr_vol_2yr_avg_annual_pct,k12_net_migr_total_2yr_avg_annual_pct,k13_net_migr_under_6_2yr_avg_annual_pct
0,01100101,Stülerstraße,3419,156,143,28,442,1162,360,377,...,59.93,40.567,-1.306,70.44,8.554,0.0,55.917,38.361,0.507,-4.83
1,01100102,Großer Tiergarten,1791,98,94,47,156,671,260,240,...,66.946,50.754,6.165,71.287,3.079,0.614,40.152,49.558,5.179,1.509
2,01100103,Lützowstraße,5211,264,306,102,604,1578,638,740,...,58.357,35.559,0.804,65.839,21.037,6.045,57.487,29.779,2.41,0.295
3,01100104,Körnerstraße,4636,256,410,112,779,1629,584,500,...,64.582,38.546,-1.399,75.937,25.295,8.24,59.032,32.693,-2.087,-2.648
4,01100205,Wilhelmstraße,2573,109,139,51,316,827,338,349,...,61.912,44.073,3.075,74.25,7.048,0.0,56.436,42.221,3.043,-5.98


In [56]:
# Nice! We have a clean dataframe now with social panel data! Let's save it to csv before we continue =)

final_df.to_csv("../data/social_panel_clean.csv", index=False)

In [57]:
# Now let's check how many missing values we have per row

missing_report = final_df.isna().agg(["sum", "mean"]).T.reset_index()
missing_report.columns = ["column_name", "missing_sum", "missing_share"]
missing_report = missing_report.sort_values("missing_share", ascending=False)
missing_report.head(40)

,column_name,missing_sum,missing_share
91,median_income_eur,2747.0,0.844711
92,median_wage_quintile,2747.0,0.844711
93,lw_ft_employee_ss_count,2725.0,0.837946
94,lw_ft_employee_ss_perc,2725.0,0.837946
90,ft_employee_ss_count,2724.0,0.837638
107,k19_male_old_age_basic_income_support_share_pct,2716.0,0.835178
106,k18_fem_old_age_basic_income_support_share_pct,2716.0,0.835178
104,k2_single_parent_hh_with_children_share_pct,2182.0,0.670972
100,d2_single_parent_child_dynamics_indicator,2181.0,0.670664
96,s2_single_parent_child_status_indicator,2181.0,0.670664
